# Multi-class image classification: ensemble comparison notebook

This notebook is a separate end-to-end ensemble workflow built to compare ensemble performance against the strong single-model baseline.

What it does:
- uses the same data loading and preprocessing recipe as the successful single-model notebook,
- trains multiple ReLU models with different seeds,
- keeps the best EMA checkpoint for each seed,
- averages their logits at validation and test time,
- compares the ensemble validation score against the single-model reference score.


In [ ]:
import copy
import os
import pickle
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Load data and keep the notebook portable

Why this step matters:
- it works in Colab and outside Colab,
- it keeps the ensemble notebook aligned with the single-model notebook,
- it fails early if the challenge files are missing.


In [ ]:
def resolve_base_path():
    candidates = []

    env_path = os.environ.get("CHALLENGE_DATA_DIR")
    if env_path:
        candidates.append(Path(env_path).expanduser())

    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd / "data",
        cwd / "dataset",
        cwd / "W2026-0-0.3",
        Path("/content/drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3"),
    ])

    def has_required_files(path):
        return (path / "train_X_y.pkl").exists() and (path / "test_X.pkl").exists()

    for candidate in candidates:
        if has_required_files(candidate):
            return candidate

    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
        drive_candidate = Path("/content/drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3")
        if has_required_files(drive_candidate):
            return drive_candidate
    except ImportError:
        pass

    checked_locations = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        "Could not find train_X_y.pkl and test_X.pkl. Checked:\n"
        f"{checked_locations}\n\n"
        "Tip: put the files in one of the locations above or set CHALLENGE_DATA_DIR."
    )


base_path = resolve_base_path()
print(f"Data directory: {base_path}")


In [ ]:
with open(base_path / "train_X_y.pkl", "rb") as f:
    X_train, y_train = pickle.load(f)

with open(base_path / "test_X.pkl", "rb") as f:
    X_test = pickle.load(f)

print("Raw X_train shape:", X_train.shape)
print("Raw y_train shape:", y_train.shape)
print("Raw X_test shape :", X_test.shape)


In [ ]:
def to_nchw(array, name):
    if array.ndim != 4:
        raise ValueError(f"{name} must be a 4D array, got shape {array.shape}")

    if array.shape[1] == 3:
        converted = array.astype(np.float32)
    elif array.shape[-1] == 3:
        converted = np.transpose(array, (0, 3, 1, 2)).astype(np.float32)
    else:
        raise ValueError(
            f"{name} must have 3 channels in either axis 1 or axis -1, got shape {array.shape}"
        )

    return converted


X_train = to_nchw(X_train, "X_train")
X_test = to_nchw(X_test, "X_test")
y_train = y_train.reshape(-1).astype(np.int64)

IMAGE_SIZE = X_train.shape[-1]
NUM_CLASSES = len(np.unique(y_train))

print("X_train shape after conversion:", X_train.shape)
print("X_test shape after conversion :", X_test.shape)
print("y_train shape after flatten   :", y_train.shape)
print("Number of classes             :", NUM_CLASSES)


## 2. Preprocessing and loaders

This keeps the ensemble setup comparable to the successful single-model run. We reuse the same train-split normalization and gentle augmentation so the comparison focuses on the effect of ensembling rather than a different preprocessing recipe.


In [ ]:
def reflect_pad_crop(x, padding):
    if padding <= 0:
        return x
    x = F.pad(x, (padding, padding, padding, padding), mode="reflect")
    top = int(torch.randint(0, 2 * padding + 1, (1,)).item())
    left = int(torch.randint(0, 2 * padding + 1, (1,)).item())
    return x[:, top:top + IMAGE_SIZE, left:left + IMAGE_SIZE]


def random_color_jitter(x, brightness=0.12, contrast=0.12, p=0.30):
    if torch.rand(1).item() >= p:
        return x

    brightness_scale = 1.0 + float(torch.empty(1).uniform_(-brightness, brightness).item())
    contrast_scale = 1.0 + float(torch.empty(1).uniform_(-contrast, contrast).item())

    channel_mean = x.mean(dim=(1, 2), keepdim=True)
    x = (x - channel_mean) * contrast_scale + channel_mean
    x = x * brightness_scale
    return torch.clamp(x, 0.0, 1.0)


class ImageDataset(Dataset):
    def __init__(self, images, labels=None, mean=None, std=None, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).float() / 255.0

        if self.augment:
            if torch.rand(1).item() < 0.5:
                x = torch.flip(x, dims=[2])
            x = reflect_pad_crop(x, padding=2)
            x = random_color_jitter(x, brightness=0.12, contrast=0.12, p=0.30)

        x = (x - self.mean) / self.std

        if self.labels is None:
            return x, torch.tensor(-1, dtype=torch.long)

        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


In [ ]:
VAL_SIZE = 0.12
BATCH_SIZE = 96
NUM_WORKERS = min(2, os.cpu_count() or 0)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train,
)

train_pixels = X_tr / 255.0
channel_mean = train_pixels.mean(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
channel_std = train_pixels.std(axis=(0, 2, 3), dtype=np.float64).astype(np.float32)
channel_std = np.clip(channel_std, 1e-6, None)

train_dataset = ImageDataset(X_tr, y_tr, mean=channel_mean, std=channel_std, augment=True)
val_dataset = ImageDataset(X_val, y_val, mean=channel_mean, std=channel_std, augment=False)
test_dataset = ImageDataset(X_test, labels=None, mean=channel_mean, std=channel_std, augment=False)

trainloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
valloader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
testloader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Channel mean:", channel_mean)
print("Channel std :", channel_std)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")
print(f"Test samples : {len(test_dataset)}")


## 3. Define the ReLU backbone used by each ensemble member

We keep the same backbone family as the strong single-model notebook so the comparison is fair. The main difference is that we now train several seeded copies of this same model and average them.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.dropout(self.bn2(self.conv2(out)))
        out = self.relu2(out + identity)
        return out


class WideSmallResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64, stride=1, dropout=0.05),
            ResidualBlock(64, 64, stride=1, dropout=0.05),
            ResidualBlock(64, 64, stride=1, dropout=0.05),
        )
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2, dropout=0.10),
            ResidualBlock(128, 128, stride=1, dropout=0.10),
            ResidualBlock(128, 128, stride=1, dropout=0.10),
        )
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2, dropout=0.15),
            ResidualBlock(256, 256, stride=1, dropout=0.15),
            ResidualBlock(256, 256, stride=1, dropout=0.15),
        )
        self.layer4 = nn.Sequential(
            ResidualBlock(256, 384, stride=1, dropout=0.20),
            ResidualBlock(384, 384, stride=1, dropout=0.20),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(384, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


net = WideSmallResNet(num_classes=NUM_CLASSES).to(device)
print(net)


## 4. Configure the ensemble and the single-model reference

The comparison target is the strong single-model result. Update `SINGLE_MODEL_REFERENCE_ACCURACY` if your best verified single-model run changes later.


In [ ]:
SINGLE_MODEL_REFERENCE_ACCURACY = 89.0
TARGET_VAL_ACCURACY = 89.0
MODEL_SEEDS = [42, 314, 2024]
EPOCHS = 50
MAX_LR = 1.2e-3
WEIGHT_DECAY = 3e-4
LABEL_SMOOTHING = 0.03
EARLY_STOPPING_PATIENCE = 12
MAX_GRAD_NORM = 1.0
MIXUP_ALPHA = 0.15
MIXUP_PROB = 0.50
EMA_DECAY = 0.995
USE_TTA = True

print(f"Single-model reference accuracy : {SINGLE_MODEL_REFERENCE_ACCURACY:.2f}%")
print(f"Ensemble seeds                 : {MODEL_SEEDS}")
print(f"Epochs per model              : {EPOCHS}")
print(f"Target validation score       : {TARGET_VAL_ACCURACY:.2f}%")


## 5. Training and evaluation helpers

Each seed gets its own model, optimizer, scheduler, scaler, and EMA tracker. We then compare both the best individual runs and the final averaged ensemble.


In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ModelEMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for param in self.shadow.parameters():
            param.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        model_state = model.state_dict()
        shadow_state = self.shadow.state_dict()
        for name, shadow_value in shadow_state.items():
            model_value = model_state[name].detach()
            if shadow_value.dtype.is_floating_point:
                shadow_value.mul_(self.decay).add_(model_value, alpha=1.0 - self.decay)
            else:
                shadow_value.copy_(model_value)


def build_training_bundle(seed):
    set_all_seeds(seed)
    model = WideSmallResNet(num_classes=NUM_CLASSES).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=MAX_LR / 12, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=MAX_LR,
        epochs=EPOCHS,
        steps_per_epoch=len(trainloader),
        pct_start=0.15,
        anneal_strategy="cos",
        div_factor=12.0,
        final_div_factor=150.0,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    ema_model = ModelEMA(model, decay=EMA_DECAY)
    return model, optimizer, scheduler, criterion, scaler, ema_model


def mixup_batch(inputs, labels, alpha=0.15, p=0.50):
    if alpha <= 0 or torch.rand(1).item() >= p:
        return inputs, labels, labels, 1.0

    lam = float(np.random.beta(alpha, alpha))
    lam = max(lam, 1.0 - lam)
    index = torch.randperm(inputs.size(0), device=inputs.device)
    mixed_inputs = lam * inputs + (1.0 - lam) * inputs[index]
    labels_b = labels[index]
    return mixed_inputs, labels, labels_b, lam


def forward_with_tta(model, inputs, use_tta=False):
    logits = model(inputs)
    if use_tta:
        flipped_inputs = torch.flip(inputs, dims=[3])
        logits = 0.5 * (logits + model(flipped_inputs))
    return logits


def run_train_epoch(model, loader, optimizer, scheduler, criterion, scaler, ema_model):
    model.train()
    running_loss = 0.0
    correct = 0.0
    total = 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        inputs, labels_a, labels_b, lam = mixup_batch(
            inputs,
            labels,
            alpha=MIXUP_ALPHA,
            p=MIXUP_PROB,
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=scaler.is_enabled()):
            logits = model(inputs)
            loss = lam * criterion(logits, labels_a) + (1.0 - lam) * criterion(logits, labels_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema_model.update(model)

        preds = torch.argmax(logits, dim=1)
        batch_correct = lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()

        running_loss += loss.item() * inputs.size(0)
        correct += batch_correct
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def evaluate_single_model(model, loader, criterion, scaler, use_tta=False):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(enabled=scaler.is_enabled()):
                logits = forward_with_tta(model, inputs, use_tta=use_tta)
                loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc, np.array(all_preds), np.array(all_labels)


def predict_logits(model_list, inputs, use_tta=False):
    logits_sum = None
    for model in model_list:
        logits = forward_with_tta(model, inputs, use_tta=use_tta)
        logits_sum = logits if logits_sum is None else logits_sum + logits
    return logits_sum / len(model_list)


def evaluate_ensemble(model_list, loader, criterion, scaler, use_tta=False):
    for model in model_list:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(enabled=scaler.is_enabled()):
                logits = predict_logits(model_list, inputs, use_tta=use_tta)
                loss = criterion(logits, labels)

            preds = torch.argmax(logits, dim=1)
            running_loss += loss.item() * inputs.size(0)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc, np.array(all_preds), np.array(all_labels)


## 6. Train the ensemble end to end

This is the longest part of the notebook. It will train one model per seed, keep the best EMA checkpoint for each, and then report both per-seed and final ensemble validation accuracy.


In [ ]:
trained_states = []
ensemble_models = []
run_histories = []
shared_scaler = GradScaler(enabled=torch.cuda.is_available())
shared_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

for model_seed in MODEL_SEEDS:
    print(f"\n{'=' * 30} Training seed {model_seed} {'=' * 30}")
    model, optimizer, scheduler, criterion, scaler, ema_model = build_training_bundle(model_seed)

    best_val_acc = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    best_state = None

    history = {
        'seed': model_seed,
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': [],
    }

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = run_train_epoch(model, trainloader, optimizer, scheduler, criterion, scaler, ema_model)
        val_loss, val_acc, _, _ = evaluate_single_model(ema_model.shadow, valloader, criterion, scaler, use_tta=USE_TTA)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {k: v.detach().cpu().clone() for k, v in ema_model.shadow.state_dict().items()}
        else:
            epochs_without_improvement += 1

        print(
            f"Seed {model_seed} | Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr:.2e}"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping seed {model_seed} at epoch {epoch}.")
            break

    trained_states.append({
        'seed': model_seed,
        'state_dict': best_state,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
    })
    run_histories.append(history)

    best_model = WideSmallResNet(num_classes=NUM_CLASSES).to(device)
    best_model.load_state_dict(best_state)
    best_model.eval()
    ensemble_models.append(best_model)

    print(f"Best validation accuracy for seed {model_seed}: {best_val_acc:.2f}% at epoch {best_epoch}")

ensemble_val_loss, ensemble_val_acc, _, _ = evaluate_ensemble(
    ensemble_models,
    valloader,
    shared_criterion,
    shared_scaler,
    use_tta=USE_TTA,
)

print("\nSingle-model best scores by seed:")
for item in trained_states:
    print(f"  Seed {item['seed']}: {item['best_val_acc']:.2f}% at epoch {item['best_epoch']}")

print(f"\nFinal ensemble validation accuracy: {ensemble_val_acc:.2f}%")
print(f"Delta vs single-model reference   : {ensemble_val_acc - SINGLE_MODEL_REFERENCE_ACCURACY:+.2f} percentage points")


In [ ]:
fig, axes = plt.subplots(len(run_histories), 3, figsize=(16, 4 * len(run_histories)))
if len(run_histories) == 1:
    axes = np.array([axes])

for row, history in enumerate(run_histories):
    epochs_ran = range(1, len(history['train_loss']) + 1)

    axes[row, 0].plot(epochs_ran, history['train_loss'], label='Train loss', color='steelblue')
    axes[row, 0].plot(epochs_ran, history['val_loss'], label='Val loss', color='darkorange')
    axes[row, 0].set_title(f"Seed {history['seed']} loss")
    axes[row, 0].set_xlabel('Epoch')
    axes[row, 0].set_ylabel('Loss')
    axes[row, 0].legend()
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].plot(epochs_ran, history['train_acc'], label='Train acc', color='seagreen')
    axes[row, 1].plot(epochs_ran, history['val_acc'], label='Val acc', color='crimson')
    axes[row, 1].axhline(y=SINGLE_MODEL_REFERENCE_ACCURACY, color='black', linestyle='--', label='single-model ref')
    axes[row, 1].set_title(f"Seed {history['seed']} accuracy")
    axes[row, 1].set_xlabel('Epoch')
    axes[row, 1].set_ylabel('Accuracy (%)')
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.3)

    axes[row, 2].plot(epochs_ran, history['lr'], color='purple')
    axes[row, 2].set_title(f"Seed {history['seed']} learning rate")
    axes[row, 2].set_xlabel('Epoch')
    axes[row, 2].set_ylabel('Learning rate')
    axes[row, 2].set_yscale('log')
    axes[row, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Validate the ensemble and compare it against the single model

This is the main comparison section. It reports the ensemble score, the gap versus the single-model baseline, and per-class behavior.


In [ ]:
val_loss, overall_acc, all_preds, all_labels = evaluate_ensemble(
    ensemble_models,
    valloader,
    shared_criterion,
    shared_scaler,
    use_tta=USE_TTA,
)

print(f"Validation loss: {val_loss:.4f}")
print(f"Overall validation accuracy: {overall_acc:.2f}%")
print(f"Single-model reference     : {SINGLE_MODEL_REFERENCE_ACCURACY:.2f}%")
print(f"Delta vs single model      : {overall_acc - SINGLE_MODEL_REFERENCE_ACCURACY:+.2f} percentage points")
print(f"Evaluation recipe          : {len(ensemble_models)}-model ensemble + EMA + flip TTA = {USE_TTA}")

print("\nPer-class accuracy:")
for cls in range(NUM_CLASSES):
    mask = all_labels == cls
    cls_count = int(mask.sum())
    cls_acc = 100.0 * np.mean(all_preds[mask] == all_labels[mask]) if cls_count > 0 else 0.0
    print(f"  Class {cls}: {cls_acc:6.2f}% ({cls_count} samples)")


## 8. Evaluate on the full training set without augmentation

This helps review whether the ensemble gain comes from genuine generalization or mostly from fitting the training set harder.


In [ ]:
full_train_eval = ImageDataset(X_train, y_train, mean=channel_mean, std=channel_std, augment=False)
full_train_loader = DataLoader(
    full_train_eval,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

train_eval_loss, train_acc, _, _ = evaluate_ensemble(
    ensemble_models,
    full_train_loader,
    shared_criterion,
    shared_scaler,
    use_tta=False,
)

print(f"Training loss without augmentation: {train_eval_loss:.4f}")
print(f"Training accuracy of the ensemble : {train_acc:.2f}%")
print(f"Gap to validation accuracy        : {train_acc - overall_acc:.2f} percentage points")


## 9. Generate test predictions with the ensemble

The submission export uses the same ensemble inference logic as validation so the comparison remains consistent end to end.


In [ ]:
def predict_loader(model_list, loader, use_tta=False):
    for model in model_list:
        model.eval()

    output_labels = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device, non_blocking=True)
            with autocast(enabled=shared_scaler.is_enabled()):
                logits = predict_logits(model_list, images, use_tta=use_tta)
            preds = torch.argmax(logits, dim=1)
            output_labels.extend(preds.cpu().numpy().tolist())

    return output_labels


output_labels = predict_loader(ensemble_models, testloader, use_tta=USE_TTA)
print(f"Generated {len(output_labels)} test predictions.")
print(f"Unique predicted classes: {sorted(set(output_labels))}")


In [ ]:
test_predictions = pd.DataFrame({
    'rowId': np.arange(len(output_labels), dtype=np.int64),
    'label': np.array(output_labels, dtype=np.int64),
})

output_csv = base_path / 'my_cnn_results_ensemble_v1.csv'
test_predictions.to_csv(output_csv, index=False)

print(test_predictions.head())
print(f"\nPredictions saved to: {output_csv}")


## 10. Save and reload the ensemble

Saving all best checkpoints preserves the exact ensemble used for the comparison.


In [ ]:
ensemble_path = Path.cwd() / 'my_cnn_model_ensemble_v1.pth'
torch.save(
    {
        'model_seeds': MODEL_SEEDS,
        'states': trained_states,
        'channel_mean': channel_mean,
        'channel_std': channel_std,
        'use_tta': USE_TTA,
        'single_model_reference_accuracy': SINGLE_MODEL_REFERENCE_ACCURACY,
    },
    ensemble_path,
)
print(f"Ensemble saved to {ensemble_path}")


In [ ]:
# To restore the ensemble later:
#
# checkpoint = torch.load(ensemble_path, map_location=device)
# reloaded_models = []
# for item in checkpoint['states']:
#     model = WideSmallResNet(num_classes=NUM_CLASSES).to(device)
#     model.load_state_dict(item['state_dict'])
#     model.eval()
#     reloaded_models.append(model)
#
# print(f"Reloaded {len(reloaded_models)} ensemble members.")
